# Ансамбли моделей машинного обучения. Часть 2.

In [8]:
import polars as pl

data = pl.read_csv("insurance.csv")
data

age,sex,bmi,children,smoker,region,charges
i64,str,f64,i64,str,str,f64
19,"""female""",27.9,0,"""yes""","""southwest""",16884.924
18,"""male""",33.77,1,"""no""","""southeast""",1725.5523
28,"""male""",33.0,3,"""no""","""southeast""",4449.462
33,"""male""",22.705,0,"""no""","""northwest""",21984.47061
32,"""male""",28.88,0,"""no""","""northwest""",3866.8552
…,…,…,…,…,…,…
50,"""male""",30.97,3,"""no""","""northwest""",10600.5483
18,"""female""",31.92,0,"""no""","""northeast""",2205.9808
18,"""female""",36.85,0,"""no""","""southeast""",1629.8335


In [9]:
data = data.with_columns(
    pl.col("sex").replace({"female": 0, "male": 1}).cast(pl.Int8),
    pl.col("smoker").replace({"no": 0, "yes": 1}).cast(pl.Int8),
)
data

age,sex,bmi,children,smoker,region,charges
i64,i8,f64,i64,i8,str,f64
19,0,27.9,0,1,"""southwest""",16884.924
18,1,33.77,1,0,"""southeast""",1725.5523
28,1,33.0,3,0,"""southeast""",4449.462
33,1,22.705,0,0,"""northwest""",21984.47061
32,1,28.88,0,0,"""northwest""",3866.8552
…,…,…,…,…,…,…
50,1,30.97,3,0,"""northwest""",10600.5483
18,0,31.92,0,0,"""northeast""",2205.9808
18,0,36.85,0,0,"""southeast""",1629.8335


In [10]:
from sklearn.preprocessing import MinMaxScaler

age_scaler = MinMaxScaler()
children_scaler = MinMaxScaler()
bmi_scaler = MinMaxScaler()
charges_scaler = MinMaxScaler()

data = data.with_columns(
    age=pl.Series(
        age_scaler.fit_transform(data.get_column("age").reshape((-1, 1)))
    ).list.explode(),
    children=pl.Series(
        children_scaler.fit_transform(data.get_column("children").reshape((-1, 1)))
    ).list.explode(),
    bmi=pl.Series(
        bmi_scaler.fit_transform(data.get_column("bmi").reshape((-1, 1)))
    ).list.explode(),
    charges=pl.Series(
        charges_scaler.fit_transform(data.get_column("charges").reshape((-1, 1)))
    ).list.explode(),
)
data

age,sex,bmi,children,smoker,region,charges
f64,i8,f64,f64,i8,str,f64
0.021739,0,0.321227,0.0,1,"""southwest""",0.251611
0.0,1,0.47915,0.2,0,"""southeast""",0.009636
0.217391,1,0.458434,0.6,0,"""southeast""",0.053115
0.326087,1,0.181464,0.0,0,"""northwest""",0.33301
0.304348,1,0.347592,0.0,0,"""northwest""",0.043816
…,…,…,…,…,…,…
0.695652,1,0.40382,0.6,0,"""northwest""",0.151299
0.0,0,0.429379,0.0,0,"""northeast""",0.017305
0.0,0,0.562012,0.0,0,"""southeast""",0.008108


In [11]:
data = data.hstack(data.get_column("region").to_dummies()).drop("region")
data

age,sex,bmi,children,smoker,charges,region_northeast,region_northwest,region_southeast,region_southwest
f64,i8,f64,f64,i8,f64,u8,u8,u8,u8
0.021739,0,0.321227,0.0,1,0.251611,0,0,0,1
0.0,1,0.47915,0.2,0,0.009636,0,0,1,0
0.217391,1,0.458434,0.6,0,0.053115,0,0,1,0
0.326087,1,0.181464,0.0,0,0.33301,0,1,0,0
0.304348,1,0.347592,0.0,0,0.043816,0,1,0,0
…,…,…,…,…,…,…,…,…,…
0.695652,1,0.40382,0.6,0,0.151299,0,1,0,0
0.0,0,0.429379,0.0,0,0.017305,1,0,0,0
0.0,0,0.562012,0.0,0,0.008108,0,0,1,0


In [12]:
y = data.get_column("charges")
X = data.drop("charges")

In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    StackingRegressor,
)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

estimators = [
    (
        "rf",
        RandomForestRegressor(
            n_estimators=300,
            max_depth=10,
            random_state=42,
        ),
    ),
    (
        "gb",
        GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=3,
            random_state=42,
        ),
    ),
]

final_estimator = LinearRegression()

stack_clf = StackingRegressor(
    estimators=estimators,
    final_estimator=final_estimator,
)

stack_clf.fit(X_train, y_train)
y_pred = stack_clf.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(stack_clf)
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")

StackingRegressor(estimators=[('rf',
                               RandomForestRegressor(max_depth=10,
                                                     n_estimators=300,
                                                     random_state=42)),
                              ('gb',
                               GradientBoostingRegressor(n_estimators=200,
                                                         random_state=42))],
                  final_estimator=LinearRegression())
RMSE: 0.0710
MAE : 0.0382
R2  : 0.8727


In [14]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    StackingRegressor,
)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    alpha=1e-3,
    max_iter=300,
    random_state=42,
)

mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(mlp)
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")

MLPRegressor(alpha=0.001, hidden_layer_sizes=(64, 32), max_iter=300,
             random_state=42)
RMSE: 0.0747
MAE : 0.0436
R2  : 0.8588


| Модель               | RMSE   | MAE    | $R^2$    |
|:-|:-:|:-:|:-:|
| StackingRegressor    | **0.0710** | **0.0382** | **0.8727** |
| Многослойный персептрон | 0.0747 | 0.0436 | 0.8588 |

По всем трём метрикам стек-регрессор заметно лучше MLP: он даёт меньшую среднюю и корневую квадратичную ошибку и выше долю объяснённой дисперсии (R²).